In [ ]:
# # Initial Imports and Variables
import numpy as np
import torch
import torchvision
import time
import matplotlib.pyplot as plt

import matplotlib
import sns
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

load_dir= "./Data/"
results_directory="./Results/"
RANDOM_STATE=2025

class_list=['Floor-Bite', 'Floor-Explore', 'Floor-Poke','Stand-Bite', 'Stand-Eat', 'Stand-Explore', 'Stand-Poke']

In [ ]:
%run Utilities.py
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip(line, cell):
    return

In [ ]:
def train_model(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes, num_epochs=25, early_stopping_patience=20, save_dir="output"):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Prepare directory to save model and history
    os.makedirs(save_dir, exist_ok=True)
    best_model_path = os.path.join(save_dir, 'best_model.pt')
    best_model_path='rubish_model.pt'
    history_path = os.path.join(save_dir, 'training_history.json')

    # Initialize training history
    history = {'train_loss': [], 'train_accuracy': [], 'val_loss': [], 'val_accuracy': [], 'lr': []}

    since = time.time()
    best_acc = 0.0
    early_stop_count = 0  # Initialize early stopping counter

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluation mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    # Check if the model is in training mode and auxiliary logits are enabled
                    if isinstance(outputs, tuple):  
                        # Extract the main output (first element of the tuple)
                        outputs = outputs[0]
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Compute epoch loss and accuracy
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Save metrics in history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_accuracy'].append(epoch_acc.item())
                history['lr'].append(optimizer.param_groups[0]['lr'])
            else:
                history['val_loss'].append(epoch_loss)
                history['val_accuracy'].append(epoch_acc.item())

            

            # Adjust learning rate based on loss (only for ReduceLROnPlateau)
            prev_lr = [group['lr'] for group in optimizer.param_groups]
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                if phase == 'val': scheduler.step(epoch_loss)
            elif phase == 'train': scheduler.step()  # safe for other schedulers like StepLR
            
            new_lr = [group['lr'] for group in optimizer.param_groups]
            
            if new_lr != prev_lr: 
                prev_str = ', '.join(f"{lr:.2e}" for lr in prev_lr)
                new_str  = ', '.join(f"{lr:.2e}" for lr in new_lr)
                print(f"📉 LR changed: {prev_str} → {new_str}")

                
            # Early stopping and model saving logic
            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    early_stop_count = 0  # Reset early stopping if improved
                    torch.save(model.state_dict(), best_model_path)
                    print(f"Best model saved with accuracy: {best_acc:.4f}")
                else:
                    early_stop_count += 1  # Increase counter if no improvement

        print()
        # Early stopping check
        if early_stop_count > early_stopping_patience:
            print("Early stopping triggered.")
            break

    # Training summary
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best validation accuracy: {best_acc:.4f}')

    # Load best model weights (now correctly saved in the persistent directory)
    model.load_state_dict(torch.load(best_model_path))
    print(f"Best model loaded from {best_model_path}")

    # Save training history
    with open(history_path, 'w') as f:
        json.dump(history, f)
    print(f"Training history saved to {history_path}")

    return model  # Now it returns at the end after all prints


# Cross Validation Rat and Behavior (Random Mislabeling of Tran/Val when choose_random)

In [ ]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
# from itertools import permutations

def read_data(k,step_size=5):
    npz_dir = load_dir
    rat_ids = ['AFH1', 'AFH2', 'AFH3', 'AFH4', 'AFH5', 'AFH6','AFL4']
    class_list = ['Floor-Bite', 'Floor-Explore', 'Floor-Poke','Stand-Bite', 'Stand-Eat', 'Stand-Explore', 'Stand-Poke']
    
    counts=np.array([[3804, 4636, 3861, 4282, 4303, 3878, 2881],
                    [1041, 3034, 2112, 1223, 2727, 2288, 1531],
                    [ 240, 4343, 1315,  486, 2013, 2122, 1888],
                    [4082, 3753,  995, 3867, 3036, 9044, 2927],
                    [ 121, 3047, 1618,   25, 2678, 1969, 2054],
                    [1810, 3720, 1048, 1219, 1750, 2591, 1215],
                    [ 540, 1922, 1339, 1012, 1612, 1840, 1245]])
    # num_rats = len(rat_ids)
    # num_behaviors = len(class_list)
    guard_frames=30
    
    x_train, y_train=[], []
    x_test, y_test=[], []
    for i,rat_id in enumerate(rat_ids):
    
        which_half=np.ones(len(class_list)) # array to select which half of the behaviors to use
        which_half[:len(which_half)//2]=0 # make half of them zero
        which_half= shuffle(which_half,random_state=RANDOM_STATE+i) ## initially it was random_state=RANDOM_STATE+i
        
        rat_data=np.load( os.path.join(npz_dir, f"{rat_id}.npz") , allow_pickle=True)
        for j, behavior in enumerate(class_list):
            data=rat_data[behavior]
            num_images = len(data)
            
            if num_images<2*guard_frames+2: # use short clips only for test
                x_test.append(data)
                y_test.append(j*np.ones(len(data),dtype=np.int64))
                continue # skip the other steps after adding the short clips to testset
                
            if num_images%2: num_images-=1
            train_idx,test_idx=np.split( np.arange(num_images),indices_or_sections=2)
            
            train_idx=train_idx[:-guard_frames]
            test_idx=test_idx[guard_frames:]
            if which_half[j]: train_idx,test_idx=test_idx,train_idx # use first half for test and first half for train
            
            x_train.append(data[train_idx[k::step_size]] ) ##### Subsampling is here
            y_train.append(j*np.ones(len(train_idx[k::step_size]),dtype=np.int64))
        
            x_test.append(data[test_idx[k::step_size]])
            y_test.append(j*np.ones(len(test_idx[k::step_size]),dtype=np.int64))
            
    
    # concatenate all arrays
    x_train = np.concatenate(x_train,axis=0 ) #[:,np.linspace(start=k, stop=train_idx.shape[1], num=num_train//train_idx.shape[0], endpoint=False,dtype=int)]
    y_train= np.concatenate( y_train,axis=0 )
    x_test = np.concatenate( x_test ,axis=0 )
    y_test= np.concatenate(  y_test ,axis=0 )
    print(x_train.shape,x_test.shape)
    
    _, train_counts = np.unique(y_train, return_counts=True)
    _, test_counts = np.unique(y_test, return_counts=True)
    print("Train class distribution:", train_counts)
    print("Test class distribution:", test_counts)
    print("difference with total", (counts[[rat_ids.index(rat) for rat in rat_ids]]-2*guard_frames).sum(0)//(2*step_size) )
    return x_train,y_train,x_test,y_test
      
                
                

def choose_random(x_train, y_train, x_test, y_test, num_train, num_test, val_size, random_state):
    trains, tests, vals = [], [], []

    for j,behavior in enumerate(class_list):
        total_trainval = int((1 + val_size) * num_train / len(class_list))
        num_train_per_class = int(num_train / len(class_list))
        num_test_per_class = int(num_test / len(class_list))

        # Train + val selection
        idx_train_all = np.where(y_train == j)[0]
        n_samples=min(len(idx_train_all), total_trainval)
        if n_samples:
            idx_train_all = shuffle(idx_train_all, random_state=random_state+j, n_samples=n_samples) ## initially it was random_state=random_state        
            train_idx = idx_train_all[:num_train_per_class]
            val_idx = idx_train_all[num_train_per_class:]
            trains.extend(train_idx)
            vals.extend(val_idx)

        # Test selection
        idx_test_all = np.where(y_test == j)[0]
        n_samples=min(len(idx_test_all), num_test_per_class)
        if n_samples:
            test_idx = shuffle(idx_test_all, random_state=random_state+j, n_samples=n_samples) ## initially it was random_state=random_state
            tests.extend(test_idx)

    return  x_train[trains], y_train[trains],x_test[tests], y_test[tests], x_train[vals], y_train[vals]

        
        
import numpy as np

def replace_percentage(arr,mis_percent,fold):
    np.random.seed(RANDOM_STATE+fold)
    arr = arr.copy()
    k = int(mis_percent*arr.size)

    # Flatten and get random indices
    idx = np.random.choice(arr.size, size=k, replace=False)
    flat = arr.ravel()

    # Generate random values
    new_vals = np.random.randint(0, len(class_list), size=k)

    # Ensure new values are different from old ones
    mask = (new_vals == flat[idx])
    while np.any(mask):
        new_vals[mask] = np.random.randint(0, len(class_list), size=np.sum(mask))
        mask = (new_vals == flat[idx])
    flat[idx] = new_vals
    return arr


In [ ]:
# %% echo skipping
import torch.nn as nn
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
from torchvision.transforms import v2
from sklearn.utils import shuffle
import torch.nn.functional as F
import copy
from itertools import permutations


def kfold_cross_validation(net, model_name,val_size=0.1,mis_percent=0):

    # Arrays to hold metrics across folds
    train_accuracies, val_accuracies, test_accuracies = [], [], []
    test_metrics, test_confusions, test_normalized_confusions = [], [], []
    
    random_choice_transform = v2.RandomChoice([
    # v2.Compose([v2.RandomHorizontalFlip(p=1.0), v2.RandomRotation(30,interpolation= transforms.InterpolationMode.NEAREST) ]),
    v2.RandomHorizontalFlip(p=1.0),
    v2.RandomRotation(30,interpolation= transforms.InterpolationMode.NEAREST),
    v2.Compose([v2.RandomZoomOut(fill=0,side_range=(1.25,1.25),p=1.0),transforms.Resize(128)]),
    v2.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 5)),
    v2.Identity(),
    ])
    # random_choice_transform=v2.RandomHorizontalFlip(p=1.0)
    
    # perms = [list(p) for p in permutations([0, 1, 2])]
    n_folds=5
    num_train,num_test=1000,5000
    
    for i in range (n_folds):
        print(f"\n📁 Fold {i + 1}/{n_folds}")
        x_train,y_train,x_test,y_test= read_data(k=i,step_size=n_folds)
        

        x_train, y_train, x_test, y_test, x_val, y_val= choose_random(x_train, y_train, x_test, y_test, num_train, num_test, val_size, RANDOM_STATE+i)
        y_train=replace_percentage(y_train,mis_percent,i) ################### before it was after x_train,y_train,x_test,y_test= read_data(k=i,step_size=n_folds)
        print("In fold",x_train.shape,y_train.shape,x_test.shape,y_test.shape,x_val.shape,y_val.shape)
            


        #################################################################################################################3
        # Create a DataLoader for training data to compute mean/std in batches
        train_dataset_raw = MyDataset(x_train, y_train,
        transform=transforms.Compose([
        transforms.ToPILImage(), 
        # transforms.Resize((299, 299), interpolation=transforms.InterpolationMode.BILINEAR), #better be before ToTensor
        transforms.ToTensor()]))
        train_loader_raw = DataLoader(train_dataset_raw, batch_size=2**13, shuffle=False)
        
        # Compute mean and std using GPU batch processing
        # mean, std = compute_mean_std_gpu(train_loader_raw)
        mean,std=[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        ################################################################################################################
        
        train_transform = transforms.Compose([
        transforms.ToPILImage(), 
        transforms.ToTensor(),
        # transforms.Resize(299),
        # random_choice_transform,
        transforms.Normalize(mean=mean, std=std)])
        
        test_transform = transforms.Compose([
        transforms.ToPILImage(), 
        transforms.ToTensor(),
        # transforms.Resize(299),
        transforms.Normalize(mean=mean, std=std)])
        
        # Create the dataset with transformations
        train_dataset = MyDataset(x_train, y_train, transform=train_transform)
        val_dataset = MyDataset(x_val, y_val, transform=test_transform)
        test_dataset = MyDataset(x_test, y_test, transform=test_transform)

        
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)
        
        # Create the DataLoader    
        dataloaders={"train": train_loader , "val": val_loader, "test": test_loader }
        dataset_sizes= {"train": len(x_train), "val": len(x_val),"test": len(x_test)}
        
        ####################################################################################
        model = copy.deepcopy(net)
        model = model.to(device)
        
        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        class_weights=torch.from_numpy(class_weights).to(device).float()
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # Observe that all parameters are being optimized
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        scheduler=ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True,threshold=0.01)
        
        # Train the model using the known train_model function
        model=train_model(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes, num_epochs=50, early_stopping_patience=20, save_dir=results_directory+model_name+"/fold_"+ str(i))
        
        # Append metrics to the arrays
        train_accuracies.append(evaluate(model, train_loader, "Train")[0][0])
        val_accuracies.append(evaluate(model, val_loader, "Validation")[0][0])
        
        metrics, confusion, normalized_confusion = evaluate(model, test_loader, "Test")
        test_accuracies.append(metrics[0])
        test_confusions.append(confusion)
        test_normalized_confusions.append(normalized_confusion)
        test_metrics.append(metrics)

        plt.figure()
        print('"accuracy", "precision","recall","f1-score"("weighted avg"),"precision","recall","f1-score" ("macro avg")+ cohens kappa+ matthews coefficient')
        print(100*metrics)
        sns.heatmap(normalized_confusion, annot=True, fmt=".2f", cmap=["vlag","icefire"][1])
        plt.title(f"Fold {i+1} - Normalized Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.show()
    
    # Average metrics across folds
    print("\nAverage Train Accuracy:", np.mean(train_accuracies))
    print("Average Validation Accuracy:", np.mean(val_accuracies))
    print("Average Test Accuracy:", np.mean(test_accuracies))
    
    
    
    test_metrics=np.vstack(test_metrics) # a matrix for which the rows are the statistics for each fold
    test_confusions= np.dstack(test_confusions) # a tensor of C*C*K
    test_normalized_confusions= np.dstack(test_normalized_confusions) # a tensor of C*C*K
    
    
    np.savez_compressed(results_directory+model_name+"/history.npz",test_metrics=test_metrics, test_confusions=test_confusions, test_normalized_confusions=test_normalized_confusions)
    print("Results saved at:", results_directory+model_name+"/history.npz", "\n")
    np.set_printoptions(precision=2, suppress=True)

    print('"accuracy", "precision","recall","f1-score"("weighted avg"),"precision","recall","f1-score" ("macro avg")+ cohens kappa+ matthews coefficient')
    print(100*test_metrics.mean(axis=0))
    print(100*test_metrics.std(axis=0))
    print("\n-------------------------------------------------------------------------------\n\n\n")



# p=np.load(results_directory)
# test_metrics0=p["test_metrics"]
# test_confusions0=p["test_confusions"]
# test_nconfusions=p["test_normalized_confusions"]

# Load the best model weights
# model.load_state_dict(torch.load(best_model_params_path))


# Do Everything

In [ ]:
import warnings
warnings.filterwarnings("ignore") # warnings.resetwarnings()	


model_list = [  (load_resnet, 0, "ResNet-18"), (load_resnet, 1,"ResNet-50"), (load_resnet, 2,"ResNet-152"),
                (load_efficientnet, 0,"EfficientNetV2_S"), (load_efficientnet, 1,"EfficientNetV2_M"),(load_efficientnet, 2,"EfficientNetV2_L"),
                (load_densenet, 0,"DenseNet-121"), (load_densenet, 1,"DenseNet-169"), (load_densenet, 2,"DenseNet-201"),
             ]

model_list=[ (load_resnet, 0, "ResNet-18")] # Set model based on your requirement. You can either of the models from the list above.
p_mises=np.flip(np.arange(0,55,5)/100) # Mislabeling Percentage
# Assuming `dataloaders` and `dataset_sizes` are already defined
for p in p_mises:
    model_func, model_index,model_name=model_list[0]
    model_name=f"Mislabeling{p}_"+model_name
    print(f"\nRunning cross-validation for model: {model_func.__name__}, model_name: {model_name}")
    kfold_cross_validation(net= model_func(model_index), model_name=model_name,mis_percent=p)